# Natura 2000 Silver → Gold: `nature2000_cell_protection`

Reads **silver_protected_site** and **silver_protected_site_h3** and builds a **gold** table
with one row per H3 cell, indicating whether it is inside a protected area and (if not)
the nearest protected area within a k-ring.

| Layer | S3 path |
|-------|---------|
| Silver in | `s3://ie-datalake/silver/nature2000/protected_site/` + `protected_site_h3/` |
| Gold out  | `s3://ie-datalake/gold/nature2000_cell_protection/country=XX/h3_resolution=N/snapshot_date=YYYY-MM-DD/` |

## Gold schema (per cell)

### Identity
- `h3_id`, `h3_resolution`, `country`, `snapshot_date`

### Protection status
- `is_protected_area` – "yes" if cell overlaps any protected site, else "no"
- `nearest_protected_distance` – grid steps to nearest protected cell (0 if protected, null if none in k-ring)

### Primary site (when cell is protected)
- `site_code`, `site_name`, `AC`, `TIPO`
- `overlap_area_km2`, `cell_area_km2`, `overlap_fraction`, `site_cover_fraction`
- `is_boundary_cell`, `is_core_cell`

### Nearest site (when cell is NOT protected, but within k-ring of protected)
- `nearest_site_code`, `nearest_site_name`, `nearest_AC`, `nearest_TIPO`
- `nearest_overlap_area_km2`, `nearest_overlap_fraction`, `nearest_site_cover_fraction`

## K-ring search
- Res 6: k=3 (max 3 grid steps)
- Res 7–9: k = 3 + (res - 6) → 4, 5, 6 (proportional to finer cells)

## Memory strategy
- Load only required columns (pyarrow projection)
- Process one (country, h3_resolution) partition at a time
- Use frozenset for O(1) protected-cell lookup

In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

S3_BUCKET = "ie-datalake"
SILVER_SITE_PREFIX = "silver/nature2000/protected_site"
SILVER_H3_PREFIX = "silver/nature2000/protected_site_h3"
GOLD_PREFIX = "gold/nature2000_cell_protection"
AWS_PROFILE = "486717354268_PowerUserAccess"

SNAPSHOT_DATE = "2026-02-27"
COUNTRIES = None  # None = all; else ["ES", "FR", ...]

H3_RESOLUTIONS = [6, 7, 8, 9]
# K-ring: res 6 → k=3; res 7→4, 8→5, 9→6 (proportional)
K_RING_BASE = 3

PARQUET_COMPRESSION = "snappy"
PARQUET_ROW_GROUP_SIZE = 100_000

In [ ]:
%pip install -q pyarrow s3fs pandas numpy h3

In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS + S3
# ─────────────────────────────────────────────────────────────────────────────

from __future__ import annotations

import logging
import os
import time
from typing import Optional

import boto3
import h3
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.fs as pafs
import pyarrow.parquet as pq
import s3fs

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("nature2000_gold")

fs = s3fs.S3FileSystem(profile=AWS_PROFILE)
_boto = boto3.Session(profile_name=AWS_PROFILE)
_creds = _boto.get_credentials().get_frozen_credentials()
_region = _boto.region_name or "eu-west-2"
fs_read = pafs.S3FileSystem(
    access_key=_creds.access_key,
    secret_key=_creds.secret_key,
    session_token=_creds.token,
    region=_region,
)

pa.set_io_thread_count(min(16, (os.cpu_count() or 4) * 2))
pa.set_cpu_count(os.cpu_count() or 4)

log.info("S3 ready (profile=%s)", AWS_PROFILE)

00:33:04 [INFO] Found credentials in shared credentials file: ~/.aws/credentials
00:33:04 [INFO] S3 ready (profile=486717354268_PowerUserAccess)


In [21]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. LOAD SILVER (projection, minimal columns)
# ─────────────────────────────────────────────────────────────────────────────

SITE_COLS = ["SITE_CODE", "SITE_NAME", "AC", "TIPO", "country", "area_km2"]
H3_COLS = ["SITE_CODE", "h3_res", "h3_id", "overlap_area_km2", "cell_area_km2",
           "overlap_fraction", "site_cover_fraction", "is_boundary_cell", "is_core_cell",
           "rank_by_overlap", "snapshot_date"]


def load_silver_sites(snapshot_date: str) -> pd.DataFrame:
    """Load protected_site for given snapshot_date. Partition cols (country) come from path."""
    base = f"{S3_BUCKET}/{SILVER_SITE_PREFIX}"
    dfs = []
    try:
        for country_dir in fs.ls(base, detail=False):
            if "country=" not in country_dir:
                continue
            country = country_dir.split("country=")[-1].rstrip("/").split("/")[0]
            part_dir = f"{country_dir}/snapshot_date={snapshot_date}"
            if not fs.exists(part_dir):
                continue
            paths = [f for f in fs.ls(part_dir, detail=False) if f.endswith(".parquet")]
            if not paths:
                continue
            dataset = ds.dataset(paths, filesystem=fs_read, format="parquet")
            avail = {c.lower(): c for c in dataset.schema.names}
            project = [avail[c.lower()] for c in SITE_COLS if c.lower() in avail and c.lower() != "country"]
            table = dataset.scanner(columns=project, use_threads=True).to_table()
            df = table.to_pandas()
            df["country"] = country
            dfs.append(df)
    except FileNotFoundError:
        log.warning("No silver protected_site found at %s", base)
        return pd.DataFrame()

    if not dfs:
        log.warning("No parquet files for snapshot_date=%s", snapshot_date)
        return pd.DataFrame()

    df = pd.concat(dfs, ignore_index=True)
    if COUNTRIES:
        df = df[df["country"].isin(COUNTRIES)]
    log.info("Loaded %d sites from silver protected_site (snapshot=%s)", len(df), snapshot_date)
    return df


def load_silver_h3(snapshot_date: str) -> pd.DataFrame:
    """Load protected_site_h3 for given snapshot."""
    base = f"{S3_BUCKET}/{SILVER_H3_PREFIX}/snapshot_date={snapshot_date}"
    info = fs_read.get_file_info(base)
    if info.type == pafs.FileType.NotFound:
        log.warning("Silver protected_site_h3 not found: %s", base)
        return pd.DataFrame()

    dataset = ds.dataset(base, filesystem=fs_read, format="parquet")
    avail = {c.lower(): c for c in dataset.schema.names}
    project = [avail[c.lower()] for c in H3_COLS if c.lower() in avail]
    table = dataset.scanner(columns=project, use_threads=True).to_table()
    df = table.to_pandas()
    log.info("Loaded %d rows from silver protected_site_h3", len(df))
    return df

In [22]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. BUILD GOLD PER (country, h3_resolution)
# ─────────────────────────────────────────────────────────────────────────────

def _k_for_res(res: int) -> int:
    return K_RING_BASE + max(0, res - 6)


def _build_nearest_protected_map(
    protected_cells: frozenset, k_max: int
) -> dict[str, tuple[str, int]]:
    """
    Precompute nearest protected cell for all cells within k_max of any protected.
    Returns dict: h3_id -> (nearest_protected_h3_id, distance). O(|protected| * disk_size).
    """
    result = {h: (h, 0) for h in protected_cells}
    for p in protected_cells:
        for k in range(1, k_max + 1):
            try:
                ring = h3.grid_ring(p, k)
            except Exception:
                continue
            for c in ring:
                c = str(c)
                if c not in result or (result[c][1] > k):
                    result[c] = (p, k)
    return result


def build_gold_partition(
    site_df: pd.DataFrame,
    h3_df: pd.DataFrame,
    country: str,
    res: int,
    snapshot_date: str,
) -> pd.DataFrame:
    """
    Build gold rows for one (country, h3_resolution).
    """
    site_country = site_df[site_df["country"] == country]
    if site_country.empty:
        return pd.DataFrame()

    site_codes = set(site_country["SITE_CODE"].astype(str))
    h3_res = h3_df[(h3_df["h3_res"] == res) & (h3_df["SITE_CODE"].astype(str).isin(site_codes))]

    if h3_res.empty:
        return pd.DataFrame()

    # Best site per (h3_id): highest overlap_fraction
    best = h3_res.sort_values("overlap_fraction", ascending=False).drop_duplicates(subset=["h3_id"], keep="first")
    protected_cells = frozenset(best["h3_id"].astype(str))

    # Build protected -> site info lookup (from best)
    site_lookup = site_country.set_index("SITE_CODE")
    site_lookup.index = site_lookup.index.astype(str)
    # h3_to_site: h3_id -> {SITE_CODE, overlap_area_km2, ...}
    best_idx = best.copy()
    best_idx["h3_id"] = best_idx["h3_id"].astype(str)
    h3_to_site = best_idx.set_index("h3_id").to_dict("index")

    # Precompute nearest protected for all cells (protected + k-ring neighbors)
    k_max = _k_for_res(res)
    nearest_map = _build_nearest_protected_map(protected_cells, k_max)

    rows = []
    for h3_id, (nearest_h3, dist) in nearest_map.items():
        is_prot = h3_id in protected_cells

        row = {
            "h3_id": h3_id,
            "h3_resolution": res,
            "country": country,
            "snapshot_date": snapshot_date,
            "is_protected_area": "yes" if is_prot else "no",
            "nearest_protected_distance": dist if dist >= 0 else None,
        }

        if is_prot:
            info = h3_to_site.get(h3_id, {})
            site_code = info.get("SITE_CODE")
            if site_code:
                sc = str(site_code)
                site_row = site_lookup.loc[sc] if sc in site_lookup.index else {}
                row["site_code"] = site_code
                row["site_name"] = site_row.get("SITE_NAME", "")
                row["AC"] = site_row.get("AC", "")
                row["TIPO"] = site_row.get("TIPO", "")
                row["overlap_area_km2"] = info.get("overlap_area_km2")
                row["cell_area_km2"] = info.get("cell_area_km2")
                row["overlap_fraction"] = info.get("overlap_fraction")
                row["site_cover_fraction"] = info.get("site_cover_fraction")
                row["is_boundary_cell"] = info.get("is_boundary_cell", False)
                row["is_core_cell"] = info.get("is_core_cell", False)
            for c in ["nearest_site_code", "nearest_site_name", "nearest_AC", "nearest_TIPO",
                      "nearest_overlap_area_km2", "nearest_overlap_fraction", "nearest_site_cover_fraction"]:
                row[c] = None
        else:
            for c in ["site_code", "site_name", "AC", "TIPO", "overlap_area_km2", "cell_area_km2",
                      "overlap_fraction", "site_cover_fraction", "is_boundary_cell", "is_core_cell"]:
                row[c] = None
            if nearest_h3 and dist >= 0:
                info = h3_to_site.get(nearest_h3, {})
                site_code = info.get("SITE_CODE")
                if site_code:
                    sc = str(site_code)
                    site_row = site_lookup.loc[sc] if sc in site_lookup.index else {}
                    row["nearest_site_code"] = site_code
                    row["nearest_site_name"] = site_row.get("SITE_NAME", "")
                    row["nearest_AC"] = site_row.get("AC", "")
                    row["nearest_TIPO"] = site_row.get("TIPO", "")
                    row["nearest_overlap_area_km2"] = info.get("overlap_area_km2")
                    row["nearest_overlap_fraction"] = info.get("overlap_fraction")
                    row["nearest_site_cover_fraction"] = info.get("site_cover_fraction")
            else:
                for c in ["nearest_site_code", "nearest_site_name", "nearest_AC", "nearest_TIPO",
                          "nearest_overlap_area_km2", "nearest_overlap_fraction", "nearest_site_cover_fraction"]:
                    row[c] = None

        rows.append(row)

    return pd.DataFrame(rows)

In [23]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. WRITE + MAIN LOOP
# ─────────────────────────────────────────────────────────────────────────────

def write_gold_partition(df: pd.DataFrame, country: str, res: int, snapshot_date: str) -> str:
    """
    Write one partition to S3. Returns S3 URI.
    Partition cols (country, h3_resolution, snapshot_date) are kept IN the parquet files
    (no partition_cols= so full schema is written).
    """
    root = f"s3://{S3_BUCKET}/{GOLD_PREFIX}/country={country}/h3_resolution={res}/snapshot_date={snapshot_date}"
    # Ensure partition cols are in df and written to parquet (not stripped)
    df = df.copy()
    df["country"] = country
    df["h3_resolution"] = res
    df["snapshot_date"] = snapshot_date
    tbl = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_to_dataset(
        tbl,
        root_path=root,
        filesystem=fs,
        existing_data_behavior="delete_matching",
        compression=PARQUET_COMPRESSION,
        row_group_size=min(PARQUET_ROW_GROUP_SIZE, len(tbl)),
    )
    return root


def run_pipeline() -> pd.DataFrame:
    """Run full silver→gold pipeline. Returns summary DataFrame."""
    t0 = time.time()
    site_df = load_silver_sites(SNAPSHOT_DATE)
    h3_df = load_silver_h3(SNAPSHOT_DATE)

    if site_df.empty or h3_df.empty:
        log.warning("Empty silver data – nothing to process")
        return pd.DataFrame()

    countries = site_df["country"].unique().tolist()
    results = []

    for country in countries:
        log.info("── %s ─────────────────────────────────────", country)
        for res in H3_RESOLUTIONS:
            t1 = time.time()
            gold = build_gold_partition(site_df, h3_df, country, res, SNAPSHOT_DATE)
            if gold.empty:
                continue
            uri = write_gold_partition(gold, country, res, SNAPSHOT_DATE)
            elapsed = time.time() - t1
            log.info("  res=%d | %d cells | %.1fs", res, len(gold), elapsed)
            results.append({
                "country": country,
                "h3_resolution": res,
                "n_cells": len(gold),
                "s3_uri": uri,
                "elapsed_s": round(elapsed, 1),
            })

    total = time.time() - t0
    log.info("✓ Pipeline complete in %.0fs", total)
    return pd.DataFrame(results)

In [24]:
# ─────────────────────────────────────────────────────────────────────────────
# EXECUTE
# ─────────────────────────────────────────────────────────────────────────────

summary = run_pipeline()
display(summary)

00:33:17 [INFO] Loaded 1862 sites from silver protected_site (snapshot=2026-02-27)
00:33:36 [INFO] Loaded 3728374 rows from silver protected_site_h3
00:33:36 [INFO] ── ES ─────────────────────────────────────
00:33:38 [INFO]   res=6 | 21985 cells | 1.9s
00:33:45 [INFO]   res=7 | 129741 cells | 6.9s
00:34:40 [INFO]   res=8 | 670728 cells | 55.2s
00:44:42 [INFO]   res=9 | 3771930 cells | 602.2s
00:44:42 [INFO] ✓ Pipeline complete in 686s


,country,h3_resolution,n_cells,s3_uri,elapsed_s
0,ES,6,21985,s3://ie-datalake/gold/nature2000_cell_protecti...,1.9
1,ES,7,129741,s3://ie-datalake/gold/nature2000_cell_protecti...,6.9
2,ES,8,670728,s3://ie-datalake/gold/nature2000_cell_protecti...,55.2
3,ES,9,3771930,s3://ie-datalake/gold/nature2000_cell_protecti...,602.2


In [28]:
# ─────────────────────────────────────────────────────────────────────────────
# PREVIEW: load gold partition h3_resolution=6 and print sample
# ─────────────────────────────────────────────────────────────────────────────

import pyarrow.dataset as ds

_part_path = f"{S3_BUCKET}/{GOLD_PREFIX}/country=ES/h3_resolution=6/snapshot_date={SNAPSHOT_DATE}"
_part_files = [f for f in fs.ls(_part_path, detail=False) if f.endswith(".parquet")]
if _part_files:
    _ds = ds.dataset(_part_files, filesystem=fs_read, format="parquet")
    _df = _ds.to_table().to_pandas()
    print(f"Loaded {len(_df):,} rows from gold h3_res=6 (ES)")
    print(_df[_df['is_protected_area'] == 'no'].head(10).to_string())
    # print(_df[["h3_id", "country", "h3_resolution", "is_protected_area", "nearest_protected_distance", "site_code", "nearest_site_code"]].head(10).to_string())
else:
    print("No gold partition found – run pipeline first")

Loaded 21,985 rows from gold h3_res=6 (ES)
                h3_id  h3_resolution country snapshot_date is_protected_area  nearest_protected_distance site_code site_name    AC  TIPO  overlap_area_km2  cell_area_km2  overlap_fraction  site_cover_fraction is_boundary_cell is_core_cell nearest_site_code                           nearest_site_name nearest_AC nearest_TIPO  nearest_overlap_area_km2  nearest_overlap_fraction  nearest_site_cover_fraction
8264  863459db7ffffff              6      ES    2026-02-27                no                           1      None      None  None  None               NaN            NaN               NaN                  NaN             None         None         ESZZ15003  Montes submarinos del suroeste de Canarias      DGBBD            B                 32.319734                  0.774602                     0.000703
8265  863458acfffffff              6      ES    2026-02-27                no                           1      None      None  None  None         